In [ ]:
from dataclasses import dataclass
from math import exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import brentq
from scipy.stats import norm


# ============================================================
# PROJECT CONFIGURATION
# ============================================================

# This matches your Windows project folder.
# The results folder is created automatically.

OUTPUT_DIR = Path(
    r"C:\Users\resma\Projects\results"
)


# ============================================================
# SECTION 1 — REPRODUCIBLE SYNTHETIC MARKET DATA
# ============================================================

def geometric_prices(
    n=1250,
    seed=42,
    initial=100.0,
    annual_drift=0.07,
    annual_vol=0.20
):

    if n < 2 or initial <= 0 or annual_vol < 0:
        raise ValueError(
            "Invalid synthetic price parameters"
        )

    rng = np.random.default_rng(seed)

    log_returns = (
        (annual_drift - 0.5 * annual_vol**2) / 252
        + annual_vol / np.sqrt(252)
        * rng.standard_normal(n)
    )

    idx = pd.bdate_range(
        "2020-01-01",
        periods=n
    )

    return pd.Series(
        initial * np.exp(np.cumsum(log_returns)),
        index=idx,
        name="asset"
    )


def cointegrated_pair(
    n=1250,
    seed=7,
    beta=1.2,
    rho=0.92
):

    """
    Generate two synthetic price series with
    a stationary AR(1) residual relationship.
    """

    if n < 2 or not -1 < rho < 1:
        raise ValueError(
            "n must be >= 2 and |rho| < 1"
        )

    rng = np.random.default_rng(seed)

    x = 4.6 + np.cumsum(
        rng.normal(0.00015, 0.012, n)
    )

    noise = rng.normal(0, 0.012, n)

    residual = np.zeros(n)

    for i in range(1, n):
        residual[i] = (
            rho * residual[i - 1] + noise[i]
        )

    y = 0.75 + beta * x + residual

    idx = pd.bdate_range(
        "2020-01-01",
        periods=n
    )

    return pd.DataFrame(
        {
            "X": np.exp(x),
            "Y": np.exp(y)
        },
        index=idx
    )


# ============================================================
# SECTION 2 — SHARED BACKTEST ENGINE
# ============================================================

@dataclass
class Backtest:
    returns: pd.Series
    equity: pd.Series
    turnover: pd.Series
    weights: pd.DataFrame
    costs: pd.Series


def run_backtest(
    prices,
    target_weights,
    fee_bps=2.0,
    slippage_bps=3.0
):

    """
    Portfolio backtesting engine.

    Signals formed at close t affect
    returns from close t to close t+1.

    Includes transaction fees and slippage.

    Simplifications:
    - No intraday execution modeling
    - No portfolio weight drift adjustment
    - No margin or borrow constraints
    """

    if fee_bps < 0 or slippage_bps < 0:
        raise ValueError(
            "Costs cannot be negative"
        )

    prices = prices.astype(float).copy()

    if (
        prices.empty
        or prices.isna().any().any()
        or not np.isfinite(prices.to_numpy()).all()
        or (prices <= 0).any().any()
    ):
        raise ValueError(
            "Prices must be positive, finite, and complete"
        )

    weights = (
        target_weights
        .reindex(
            index=prices.index,
            columns=prices.columns
        )
        .fillna(0.0)
        .astype(float)
    )

    if not np.isfinite(weights.to_numpy()).all():
        raise ValueError(
            "Target weights must be finite"
        )

    # Delay holdings by one bar to avoid look-ahead.
    held = weights.shift(1).fillna(0.0)

    asset_returns = (
        prices.pct_change().fillna(0.0)
    )

    gross = (
        held * asset_returns
    ).sum(axis=1)

    turnover = (
        held.diff().abs().sum(axis=1)
    )

    if len(turnover):
        turnover.iloc[0] = (
            held.iloc[0].abs().sum()
        )

    costs = (
        turnover
        * (fee_bps + slippage_bps)
        / 10_000
    )

    net = (
        gross - costs
    ).rename("net_return")

    equity = (
        1 + net
    ).cumprod().rename("equity")

    return Backtest(
        returns=net,
        equity=equity,
        turnover=turnover,
        weights=held,
        costs=costs
    )


# ============================================================
# SECTION 3 — PERFORMANCE AND RISK ANALYTICS
# ============================================================

def metrics(
    returns,
    periods=252
):

    r = returns.dropna().astype(float)

    if len(r) < 2:
        raise ValueError(
            "Need at least two returns"
        )

    equity = (1 + r).cumprod()

    drawdown = (
        equity
        / equity.cummax().clip(lower=1)
        - 1
    )

    sigma = float(
        r.std(ddof=1)
    )

    downside = float(
        np.sqrt(
            np.mean(
                np.minimum(
                    r.to_numpy(),
                    0
                ) ** 2
            )
        )
    )

    total_equity = float(
        equity.iloc[-1]
    )

    sharpe = (
        float(
            r.mean() / sigma * np.sqrt(periods)
        )
        if sigma > 0
        else None
    )

    sortino = (
        float(
            r.mean() / downside * np.sqrt(periods)
        )
        if downside > 0
        else None
    )

    cagr = (
        total_equity ** (periods / len(r)) - 1
        if total_equity > 0
        else -1.0
    )

    return {
        "total_return": total_equity - 1,
        "cagr": cagr,
        "annual_volatility": sigma * np.sqrt(periods),
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": float(drawdown.min()),
        "hit_rate": float((r > 0).mean())
    }


def block_bootstrap_sharpe_ci(
    returns,
    n_boot=500,
    block=10,
    seed=42
):

    """
    Moving-block bootstrap for an approximate
    95% confidence interval around the Sharpe ratio.
    """

    r = (
        returns.dropna()
        .to_numpy(dtype=float)
    )

    if (
        block < 1
        or len(r) < block
        or n_boot < 10
    ):
        raise ValueError(
            "Invalid bootstrap parameters"
        )

    rng = np.random.default_rng(seed)

    sharpes = []

    for _ in range(n_boot):

        starts = rng.integers(
            0,
            len(r) - block + 1,
            size=int(
                np.ceil(len(r) / block)
            )
        )

        sample = np.concatenate(
            [
                r[s:s + block]
                for s in starts
            ]
        )[:len(r)]

        sd = sample.std(ddof=1)

        sharpes.append(
            float(
                sample.mean()
                / sd
                * np.sqrt(252)
            )
            if sd > 0
            else np.nan
        )

    if np.all(np.isnan(sharpes)):
        return (None, None)

    return tuple(
        float(x)
        for x in np.nanpercentile(
            sharpes,
            [2.5, 97.5]
        )
    )


# ============================================================
# SECTION 4 — MODEL 1: MOMENTUM BACKTEST
# ============================================================

def momentum_weights(
    prices,
    lookback=126,
    long_short=True
):

    if lookback < 2:
        raise ValueError(
            "Momentum lookback must be >= 2"
        )

    signal = (
        prices / prices.shift(lookback)
        - 1
    )

    position = (
        np.sign(signal)
        if long_short
        else (signal > 0).astype(float)
    )

    position = pd.Series(
        position,
        index=prices.index
    ).where(
        signal.notna(),
        0.0
    )

    return pd.DataFrame({
        prices.name or "asset": position
    })


def run_momentum(
    prices,
    lookback=126,
    fee_bps=2,
    slippage_bps=3
):

    prices = prices.rename(
        prices.name or "asset"
    )

    return run_backtest(
        prices.to_frame(),
        momentum_weights(
            prices,
            lookback
        ),
        fee_bps,
        slippage_bps
    )


@dataclass
class WalkForwardResult:
    selected_lookback: int
    train_scores: dict
    test_backtest: Backtest
    train_end: object
    test_start: object


def walk_forward_momentum(
    prices,
    lookbacks=(21, 63, 126, 252),
    train_fraction=0.65,
    embargo_days=5
):

    """
    Select the momentum lookback using training
    data, then evaluate on a held-out test period.

    Uses a single train/test split.
    """

    if (
        not 0.3 < train_fraction < 0.9
        or embargo_days < 0
    ):
        raise ValueError(
            "Invalid train fraction or embargo"
        )

    cut = int(
        len(prices) * train_fraction
    )

    candidates = tuple(
        int(lb)
        for lb in lookbacks
    )

    if (
        not candidates
        or min(candidates) < 2
        or max(candidates) + 30 >= cut
        or cut + embargo_days + 30 >= len(prices)
    ):
        raise ValueError(
            "Insufficient training or testing data"
        )

    scores = {}

    for lb in candidates:

        bt = run_momentum(
            prices.iloc[:cut],
            lb
        )

        score = metrics(
            bt.returns.iloc[lb + 1:]
        )["sharpe"]

        scores[lb] = (
            float(score)
            if score is not None
            and np.isfinite(score)
            else None
        )

    chosen = max(
        candidates,
        key=lambda lb: (
            scores[lb]
            if scores[lb] is not None
            else -np.inf
        )
    )

    bt = run_momentum(
        prices,
        chosen
    )

    ix = cut + embargo_days

    test_returns = (
        bt.returns.iloc[ix:]
    )

    test_bt = Backtest(
        returns=test_returns,
        equity=(1 + test_returns).cumprod(),
        turnover=bt.turnover.iloc[ix:],
        weights=bt.weights.iloc[ix:],
        costs=bt.costs.iloc[ix:]
    )

    return WalkForwardResult(
        selected_lookback=chosen,
        train_scores=scores,
        test_backtest=test_bt,
        train_end=prices.index[cut - 1],
        test_start=prices.index[ix]
    )


# ============================================================
# SECTION 5 — MODEL 2: OPTIONS PRICING
# ============================================================

@dataclass(frozen=True)
class Greeks:
    price: float
    delta: float
    gamma: float
    vega: float
    theta: float
    rho: float


def black_scholes(
    spot,
    strike,
    maturity,
    rate,
    sigma,
    kind="call",
    dividend=0
):

    """
    Black-Scholes-Merton European option pricer.

    Returns option price and Greeks:
    delta, gamma, vega, theta, and rho.
    """

    if (
        min(spot, strike, maturity, sigma) <= 0
        or kind not in ("call", "put")
    ):
        raise ValueError(
            "Invalid European option inputs"
        )

    d1 = (
        log(spot / strike)
        + (
            rate
            - dividend
            + sigma**2 / 2
        ) * maturity
    ) / (
        sigma * sqrt(maturity)
    )

    d2 = (
        d1 - sigma * sqrt(maturity)
    )

    df_r = exp(
        -rate * maturity
    )

    df_q = exp(
        -dividend * maturity
    )

    sign = (
        1 if kind == "call" else -1
    )

    price = sign * (
        spot * df_q * norm.cdf(sign * d1)
        - strike * df_r * norm.cdf(sign * d2)
    )

    delta = (
        sign * df_q * norm.cdf(sign * d1)
    )

    gamma = (
        df_q * norm.pdf(d1)
        / (spot * sigma * sqrt(maturity))
    )

    vega = (
        spot * df_q * norm.pdf(d1) * sqrt(maturity)
    )

    theta = (
        -spot * df_q * norm.pdf(d1)
        * sigma / (2 * sqrt(maturity))
        - sign * rate * strike * df_r
        * norm.cdf(sign * d2)
        + sign * dividend * spot * df_q
        * norm.cdf(sign * d1)
    )

    rho = (
        sign * strike * maturity * df_r
        * norm.cdf(sign * d2)
    )

    return Greeks(
        *(
            float(x)
            for x in (
                price,
                delta,
                gamma,
                vega,
                theta,
                rho
            )
        )
    )


def implied_vol(
    market_price,
    spot,
    strike,
    maturity,
    rate,
    kind="call",
    dividend=0
):

    """
    Calculate implied volatility using
    Brent's root-finding algorithm.
    """

    if (
        min(spot, strike, maturity) <= 0
        or kind not in ("call", "put")
    ):
        raise ValueError(
            "Invalid implied volatility inputs"
        )

    discounted_spot = (
        spot * exp(-dividend * maturity)
    )

    discounted_strike = (
        strike * exp(-rate * maturity)
    )

    intrinsic = (
        discounted_spot - discounted_strike
        if kind == "call"
        else discounted_strike - discounted_spot
    )

    lower = max(
        0.0,
        intrinsic
    )

    upper = (
        discounted_spot
        if kind == "call"
        else discounted_strike
    )

    if not lower < market_price < upper:
        raise ValueError(
            "Option price violates strict no-arbitrage bounds"
        )

    def objective(v):

        return (
            black_scholes(
                spot,
                strike,
                maturity,
                rate,
                v,
                kind,
                dividend
            ).price - market_price
        )

    low = 1e-8
    high = 1.0

    while (
        objective(high) < 0
        and high < 64
    ):
        high *= 2

    if (
        objective(low) * objective(high)
        > 0
    ):
        raise ValueError(
            "Cannot bracket implied volatility root"
        )

    return float(
        brentq(
            objective,
            low,
            high,
            xtol=1e-12
        )
    )


def monte_carlo_european(
    spot,
    strike,
    maturity,
    rate,
    sigma,
    kind="call",
    dividend=0,
    n_paths=100_000,
    seed=42
):

    """
    European option Monte Carlo simulation.

    Includes:
    - Antithetic random variables
    - Control variates
    - 95% confidence interval
    """

    if (
        min(spot, strike, maturity, sigma) <= 0
        or kind not in ("call", "put")
    ):
        raise ValueError(
            "Invalid Monte Carlo option inputs"
        )

    if (
        not isinstance(n_paths, int)
        or n_paths < 100
        or n_paths % 2
    ):
        raise ValueError(
            "n_paths must be an even integer >= 100"
        )

    rng = np.random.default_rng(seed)

    half = n_paths // 2

    z_half = rng.standard_normal(half)

    z = np.concatenate(
        (z_half, -z_half)
    )

    terminal = spot * np.exp(
        (
            rate
            - dividend
            - sigma**2 / 2
        ) * maturity
        + sigma * np.sqrt(maturity) * z
    )

    sign = (
        1 if kind == "call" else -1
    )

    discount = np.exp(
        -rate * maturity
    )

    payoff = (
        discount
        * np.maximum(
            sign * (terminal - strike),
            0
        )
    )

    control = (
        discount * terminal
    )

    known_mean = (
        spot * np.exp(-dividend * maturity)
    )

    coefficient = (
        np.cov(
            payoff,
            control,
            ddof=1
        )[0, 1]
        / np.var(
            control,
            ddof=1
        )
    )

    adjusted = (
        payoff
        - coefficient * (control - known_mean)
    )

    # Pair each draw with its antithetic counterpart.
    paired = (
        adjusted[:half]
        + adjusted[half:]
    ) / 2

    price = float(
        paired.mean()
    )

    standard_error = float(
        paired.std(ddof=1)
        / np.sqrt(half)
    )

    confidence_interval = (
        price - 1.96 * standard_error,
        price + 1.96 * standard_error
    )

    return (
        price,
        confidence_interval
    )


# ============================================================
# SECTION 6 — MODEL 3: VOLATILITY TARGETING
# ============================================================

def volatility_target_weights(
    prices,
    annual_target=0.12,
    ewma_lambda=0.94,
    max_leverage=1.5,
    min_history=30
):

    """
    Dynamically adjust exposure according
    to exponentially weighted volatility.
    """

    if (
        annual_target <= 0
        or not 0 < ewma_lambda < 1
        or max_leverage <= 0
        or min_history < 2
    ):
        raise ValueError(
            "Invalid volatility targeting parameters"
        )

    r = prices.pct_change()

    variance = (
        r.pow(2).ewm(
            alpha=1 - ewma_lambda,
            adjust=False,
            min_periods=min_history
        ).mean()
    )

    forecast = np.sqrt(
        variance * 252
    )

    exposure = (
        annual_target
        / forecast.replace(0, np.nan)
    ).clip(
        upper=max_leverage
    )

    exposure = exposure.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

    return pd.DataFrame({
        prices.name or "asset": exposure
    })


def run_volatility_target(
    prices,
    annual_target=0.12,
    max_leverage=1.5,
    fee_bps=2,
    slippage_bps=3
):

    p = prices.rename(
        prices.name or "asset"
    )

    w = volatility_target_weights(
        p,
        annual_target,
        max_leverage=max_leverage
    )

    return run_backtest(
        p.to_frame(),
        w,
        fee_bps,
        slippage_bps
    )


def realized_vol_by_regime(
    backtest,
    prices,
    lookback=30
):

    """
    Analyze strategy risk during high-
    and low-volatility market conditions.
    """

    asset_vol = (
        prices.pct_change()
        .rolling(lookback)
        .std()
        * np.sqrt(252)
    )

    median = asset_vol.median()

    high = asset_vol > median

    aligned = (
        backtest.returns.reindex(
            asset_vol.index
        )
    )

    return {
        "high_vol": float(
            aligned[high].std() * np.sqrt(252)
        ),
        "low_vol": float(
            aligned[
                ~high & asset_vol.notna()
            ].std() * np.sqrt(252)
        )
    }


# ============================================================
# SECTION 7 — MODEL 4: PAIRS TRADING
# ============================================================

def pairs_weights(
    prices,
    hedge_window=126,
    z_window=30,
    entry=2.0,
    exit=0.5
):

    """
    Statistical arbitrage strategy using
    rolling regression and z-score signals.
    """

    if (
        prices.shape[1] != 2
        or hedge_window < 10
        or z_window < 5
        or entry <= exit
        or exit < 0
    ):
        raise ValueError(
            "Require two assets and valid strategy parameters"
        )

    a, b = prices.columns

    x = np.log(
        prices[b]
    )

    y = np.log(
        prices[a]
    )

    beta = (
        x.rolling(
            hedge_window
        ).cov(y)
        / x.rolling(
            hedge_window
        ).var().replace(0, np.nan)
    )

    beta = beta.replace(
        [np.inf, -np.inf],
        np.nan
    )

    alpha = (
        y.rolling(hedge_window).mean()
        - beta * x.rolling(hedge_window).mean()
    )

    spread = (
        y - (alpha + beta * x)
    )

    center = (
        spread.rolling(z_window).mean()
    )

    std = (
        spread.rolling(z_window)
        .std()
        .replace(0, np.nan)
    )

    z = (
        spread - center
    ) / std

    state = 0
    states = []

    for score in z:

        if not np.isfinite(score):
            state = 0

        elif state == 0:

            if score > entry:
                state = -1

            elif score < -entry:
                state = 1

        elif abs(score) < exit:
            state = 0

        states.append(state)

    position = pd.Series(
        states,
        index=prices.index
    )

    denominator = (
        1 + beta.abs()
    )

    weights = pd.DataFrame(
        {
            a: position / denominator,
            b: -position * beta / denominator
        },
        index=prices.index
    ).fillna(0)

    diagnostics = pd.DataFrame(
        {
            "zscore": z,
            "beta": beta,
            "spread": spread,
            "position": position
        }
    )

    return (
        weights,
        diagnostics
    )


def run_pairs(
    prices,
    **kwargs
):

    weights, diagnostics = pairs_weights(
        prices,
        **kwargs
    )

    backtest = run_backtest(
        prices,
        weights
    )

    return (
        backtest,
        diagnostics
    )


def half_life(
    spread
):

    """
    Estimate a spread's mean-reversion
    half-life in bars using AR(1).

    Returns None when a reverting estimate
    is not available.
    """

    s = (
        spread.replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
        .to_numpy()
    )

    if len(s) < 30:
        raise ValueError(
            "Need at least 30 spread observations"
        )

    phi = np.polyfit(
        s[:-1],
        s[1:],
        1
    )[0]

    if not 0 < phi < 1:
        return None

    return float(
        -np.log(2) / np.log(phi)
    )


# ============================================================
# SECTION 8 — MODEL 5: MARKET MAKING SIMULATOR
# ============================================================

@dataclass
class MarketMakingResult:
    events: pd.DataFrame
    summary: dict


def simulate_market_maker(
    n_steps=10_000,
    seed=42,
    initial_mid=100,
    initial_cash=100_000,
    sigma_step=0.04,
    half_spread=0.04,
    inventory_skew=0.008,
    inventory_limit=30,
    base_fill_probability=0.12,
    adverse_selection=0.012
):

    """
    Inventory-aware market making.

    Features:
    - Bid/ask quote generation
    - Inventory skew
    - Probabilistic fills
    - Inventory limits
    - Adverse selection
    - Mark-to-market profit and loss

    This is a simplified stochastic simulator,
    not a full limit order book.
    """

    if (
        n_steps < 2
        or min(
            initial_mid,
            initial_cash,
            sigma_step,
            half_spread
        ) <= 0
        or not 0 < base_fill_probability < 1
        or inventory_limit < 1
        or inventory_skew < 0
        or adverse_selection < 0
    ):
        raise ValueError(
            "Invalid market-making parameters"
        )

    rng = np.random.default_rng(seed)

    mid = float(initial_mid)

    cash = float(initial_cash)

    inventory = 0

    pending_drift = 0.0

    fills = 0
    buy_count = 0
    sell_count = 0

    rows = []

    for step in range(n_steps):

        # Simulate the next market midpoint.
        mid = max(
            0.01,
            mid
            + sigma_step * rng.standard_normal()
            + pending_drift
        )

        # Inventory-aware quote adjustment.
        reservation = (
            mid - inventory_skew * inventory
        )

        bid = (
            reservation - half_spread
        )

        ask = (
            reservation + half_spread
        )

        # Enforce inventory limits.
        bid_allowed = (
            inventory < inventory_limit
            and bid > 0
        )

        ask_allowed = (
            inventory > -inventory_limit
        )

        # Probability of execution.
        p_bid = (
            base_fill_probability
            * np.exp(
                -max(mid - bid, 0) / half_spread
            )
        )

        p_ask = (
            base_fill_probability
            * np.exp(
                -max(ask - mid, 0) / half_spread
            )
        )

        bought = bool(
            bid_allowed
            and rng.random() < p_bid
        )

        sold = bool(
            ask_allowed
            and rng.random() < p_ask
        )

        # Buy execution at bid.
        if bought:
            inventory += 1
            cash -= bid
            buy_count += 1
            fills += 1

        # Sell execution at ask.
        if sold:
            inventory -= 1
            cash += ask
            sell_count += 1
            fills += 1

        # Apply adverse selection on next tick.
        pending_drift = (
            adverse_selection
            * (int(sold) - int(bought))
        )

        equity = (
            cash + inventory * mid
        )

        rows.append(
            (
                step,
                mid,
                bid,
                ask,
                inventory,
                cash,
                equity,
                int(bought),
                int(sold)
            )
        )

    events = pd.DataFrame(
        rows,
        columns=[
            "step",
            "mid",
            "bid",
            "ask",
            "inventory",
            "cash",
            "equity",
            "buy_fill",
            "sell_fill"
        ]
    ).set_index("step")

    pnl = float(
        events.equity.iloc[-1] - initial_cash
    )

    drawdown = (
        events.equity
        / events.equity.cummax()
        - 1
    )

    summary = {
        "pnl": pnl,
        "return_on_initial_cash": (
            pnl / initial_cash
        ),
        "max_drawdown": float(
            drawdown.min()
        ),
        "fills": fills,
        "buy_fills": buy_count,
        "sell_fills": sell_count,
        "final_inventory": inventory,
        "max_abs_inventory": int(
            events.inventory.abs().max()
        ),
        "inventory_limit": inventory_limit
    }

    return MarketMakingResult(
        events=events,
        summary=summary
    )


# ============================================================
# SECTION 9 — CHART GENERATION
# ============================================================

def plot_series(
    series,
    title,
    path,
    show=True
):

    fig, ax = plt.subplots(
        figsize=(9, 4)
    )

    series.plot(ax=ax)

    ax.set(
        title=title,
        xlabel="Date / step",
        ylabel="Value"
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    fig.savefig(
        path,
        dpi=140
    )

    # Show charts directly inside the notebook.
    if show:
        plt.show()

    plt.close(fig)


# ============================================================
# SECTION 10 — SAFE JSON REPORTING
# ============================================================

def json_safe(value):

    """
    Convert NumPy and non-finite values
    to JSON-compatible Python values.
    """

    if isinstance(value, dict):

        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (tuple, list)):

        return [
            json_safe(v)
            for v in value
        ]

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(
        value,
        (float, np.floating)
    ):

        if np.isfinite(value):
            return float(value)

        return None

    return value


# ============================================================
# SECTION 11 — MAIN QUANTITATIVE RESEARCH PIPELINE
# ============================================================

def main(
    output_dir=OUTPUT_DIR,
    seed=42,
    show_charts=True
):

    """
    Execute all five models and save:
    - Six PNG charts
    - JSON research report
    """

    output = Path(output_dir)

    output.mkdir(
        parents=True,
        exist_ok=True
    )

    print("=" * 65)
    print("QUANT INTERVIEW LAB")
    print("=" * 65)

    print("\nGenerating synthetic market data...")

    prices = geometric_prices(
        seed=seed
    )

    # --------------------------------------------------------
    # MODEL 1 — MOMENTUM
    # --------------------------------------------------------

    print("\n[1/5] Running momentum backtest...")

    momentum = walk_forward_momentum(
        prices
    )

    momentum_report = {
        "selected_lookback": (
            momentum.selected_lookback
        ),
        "train_sharpes": (
            momentum.train_scores
        ),
        "train_end": str(
            momentum.train_end.date()
        ),
        "test_start": str(
            momentum.test_start.date()
        ),
        "test_metrics": metrics(
            momentum.test_backtest.returns
        ),
        "test_sharpe_95pct_ci": (
            block_bootstrap_sharpe_ci(
                momentum.test_backtest.returns
            )
        )
    }

    plot_series(
        momentum.test_backtest.equity,
        "Momentum: Out-of-Sample Net Equity",
        output / "momentum.png",
        show_charts
    )

    # --------------------------------------------------------
    # MODEL 2 — OPTIONS PRICING
    # --------------------------------------------------------

    print("\n[2/5] Pricing European options...")

    option = black_scholes(
        100,
        105,
        0.75,
        0.04,
        0.25,
        dividend=0.01
    )

    mc_price, mc_ci = monte_carlo_european(
        100,
        105,
        0.75,
        0.04,
        0.25,
        dividend=0.01,
        seed=seed
    )

    options_report = {
        "black_scholes": vars(option),
        "implied_volatility": implied_vol(
            option.price,
            100,
            105,
            0.75,
            0.04,
            dividend=0.01
        ),
        "monte_carlo_price": mc_price,
        "monte_carlo_95pct_ci": mc_ci
    }

    # --------------------------------------------------------
    # MODEL 3 — VOLATILITY TARGETING
    # --------------------------------------------------------

    print("\n[3/5] Running volatility targeting...")

    vol = run_volatility_target(
        prices
    )

    volatility_report = {
        "metrics": metrics(
            vol.returns
        ),
        "volatility_regimes": (
            realized_vol_by_regime(
                vol,
                prices
            )
        ),
        "mean_daily_turnover": float(
            vol.turnover.mean()
        )
    }

    plot_series(
        vol.equity,
        "Volatility Targeting: Net Equity",
        output / "vol_target.png",
        show_charts
    )

    # --------------------------------------------------------
    # MODEL 4 — PAIRS TRADING
    # --------------------------------------------------------

    print("\n[4/5] Running pairs trading...")

    pair_prices = cointegrated_pair(
        seed=seed
    )

    pairs_bt, diag = run_pairs(
        pair_prices
    )

    pairs_report = {
        "metrics": metrics(
            pairs_bt.returns
        ),
        "estimated_spread_half_life_bars": (
            half_life(diag.spread)
        ),
        "mean_daily_turnover": float(
            pairs_bt.turnover.mean()
        )
    }

    plot_series(
        pairs_bt.equity,
        "Pairs Trading: Net Equity",
        output / "pairs.png",
        show_charts
    )

    plot_series(
        diag.zscore,
        "Pairs Trading: Z-Score",
        output / "pairs_zscore.png",
        show_charts
    )

    # --------------------------------------------------------
    # MODEL 5 — MARKET MAKING
    # --------------------------------------------------------

    print("\n[5/5] Simulating market making...")

    mm = simulate_market_maker(
        seed=seed
    )

    plot_series(
        mm.events.equity,
        "Market Maker: Mark-to-Market Equity",
        output / "market_maker.png",
        show_charts
    )

    plot_series(
        mm.events.inventory,
        "Market Maker: Inventory",
        output / "market_maker_inventory.png",
        show_charts
    )

    # --------------------------------------------------------
    # COMBINE REPORTS
    # --------------------------------------------------------

    report = json_safe(
        {
            "momentum": momentum_report,
            "options": options_report,
            "volatility_targeting": volatility_report,
            "pairs": pairs_report,
            "market_making": mm.summary,
            "disclaimer": (
                "Synthetic data and simplified fills; "
                "educational use only."
            )
        }
    )

    report_path = (
        output / "report.json"
    )

    report_path.write_text(
        json.dumps(
            report,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # PRINT FINAL RESULTS
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("ALL FIVE MODELS COMPLETED")
    print("=" * 65)

    print(
        f"\nSaved six charts and report to: "
        f"{output.resolve()}"
    )

    print(
        json.dumps(
            report,
            indent=2,
            allow_nan=False
        )
    )

    return report


# ============================================================
# SECTION 12 — RUN ALL FIVE MODELS IN JUPYTER
# ============================================================

# Run everything and display the charts.
# No argparse or command-line arguments are needed.

report = main(
    output_dir=OUTPUT_DIR,
    seed=42,
    show_charts=True
)



AttributeError: partially initialized module 'pandas' from 'C:\Users\resma\AppData\Roaming\Python\Python314\site-packages\pandas\__init__.py' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)